# EfficientNet-B0 Training

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# set random seeds for reproducibility
RANDOM_STATE = 8
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# check what device we're using
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"Using device: CUDA")
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print(f"Using device: MPS (Apple Silicon)")
else:
    DEVICE = torch.device('cpu')
    print(f"Using device: CPU")

print(f"PyTorch version: {torch.__version__}")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Using device: MPS (Apple Silicon)
PyTorch version: 2.6.0


## Configuration

Hyperparameters, etc

In [2]:
# paths and basic setup
DATA_DIR = Path('../../data')
DATASET_DIR = DATA_DIR / 'shark_dataset_split'
IMAGE_SIZE = 224

# training hyperparameters
BATCH_SIZE = 32
EPOCHS = 150
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.01
PATIENCE = 25

# focal loss parameters
FOCAL_ALPHA = 1.0
FOCAL_GAMMA = 2.0

## Data Augmentation

The images are melting curve graphs with temperature on the x-axis and fluorescence on the y-axis, so transformations were carefully chosen.
- resize to 224x224
- small vertical shifts
- small brightness and contrast jitter
- small gaussian noise
- normalize to the standard ImageNet normalization

In [3]:
class AddGaussianNoise(object):
    def __init__(self, std=0.005):
        self.std = std
    
    def __call__(self, tensor):
        # add a small random value to each pixel in the image
        return tensor + torch.randn_like(tensor) * self.std

def get_transforms(is_training=True):
    if is_training:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomAffine(degrees=0, translate=(0.0, 0.03)), # small vertical shift only
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            AddGaussianNoise(std=0.005), # measurement noise
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        # no augmentation for validation/test
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

## Model Architecture

EfficientNet-B0 pretrained on ImageNet, then fine-tuned for the 57 shark species

In [4]:
def get_model(num_classes):
    model = models.efficientnet_b0(weights='IMAGENET1K_V1')
    
    # replace the classifier head to have the correct amount of output nodes
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes) # resize output layer
    )
    
    return model

## Loss Function

Focal loss is better than CE when there is a class imbalance.

In [5]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

## Training Functions

Training loop

In [6]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # forward pass
        optimizer.zero_grad()
        outputs = model(inputs)

        # measure loss
        loss = criterion(outputs, labels)
        
        # backward pass 
        loss.backward()
        optimizer.step()
        
        # accumulate metrics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(loader), 100. * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # predict
            outputs = model(inputs)

            # measure loss
            loss = criterion(outputs, labels)
            
            # accumulate metrics
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(loader), 100. * correct / total

## Load Dataset

In [7]:
# Load train, val, and test datasets
train_transform = get_transforms(is_training=True)
val_transform = get_transforms(is_training=False)
test_transform = get_transforms(is_training=False)

train_dataset = datasets.ImageFolder(DATA_DIR / 'train', transform=train_transform)
val_dataset = datasets.ImageFolder(DATA_DIR / 'val', transform=val_transform)
test_dataset = datasets.ImageFolder(DATA_DIR / 'test', transform=test_transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

num_classes = len(train_dataset.classes)
class_names = train_dataset.classes

print(f"Train set: {len(train_dataset)} samples")
print(f"Val set: {len(val_dataset)} samples")
print(f"Test set: {len(test_dataset)} samples")
print(f"Number of classes: {num_classes}")
print(f"Class names: {class_names[:5]}... (showing first 5)")

Train set: 385 samples
Val set: 129 samples
Test set: 129 samples
Number of classes: 55
Class names: ['Arabian_smooth-hound', 'Atlantic_Sharpnose_shark', 'Blackchin_guitarfish', 'Blacknose_shark', 'Blackspotted_smooth-hound']... (showing first 5)


## Training

Train model on train set with validation

In [8]:
# Initialize model
model = get_model(num_classes).to(DEVICE)

# Loss, optimizer, scheduler
criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# Tracking
best_val_acc = 0.0
best_model_state = None
patience_counter = 0

# History for training
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'learning_rates': []
}

print(f"{'='*60}")
print("Training EfficientNet-B0")
print('='*60)

# Training loop
for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = validate(model, val_loader, criterion, DEVICE)
    
    # Store metrics
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['learning_rates'].append(optimizer.param_groups[0]['lr'])
    
    scheduler.step()
    
    # Check if this is the best model so far
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1
    
    # Print progress every 10 epochs
    if epoch % 10 == 0 or epoch == EPOCHS - 1:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | "
              f"Best: {best_val_acc:.2f}%")
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\n{'='*60}")
print(f"Training completed. Best validation accuracy: {best_val_acc:.2f}%")
print('='*60)

# Load best model
if best_model_state:
    model.load_state_dict(best_model_state)

# Save the model
acc_str = f"{best_val_acc:.2f}".replace('.', '')
model_filename = f'efficientnet_b0_{acc_str}.pth'
torch.save({
    'model_state_dict': best_model_state,
    'val_acc': best_val_acc,
    'history': history,
    'class_names': class_names
}, model_filename)
print(f"Model saved to {model_filename}")

Training EfficientNet-B0
Epoch   1/150 | Train Loss: 3.7530 Acc: 7.53% | Val Loss: 3.6720 Acc: 6.20% | Best: 6.20%


KeyboardInterrupt: 

## Test Set Evaluation

Evaluate on the test set.

In [ ]:
def test_model(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            # forward pass
            outputs = torch.softmax(model(images), dim=1)
            _, predicted = outputs.max(1)
            
            # accumulate statistics
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            # store predictions
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return 100. * correct / total, all_preds, all_labels

# Test the model
test_acc, all_preds, all_labels = test_model(model, test_loader, DEVICE)

print(f"\nTest Set Results:")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Total samples: {len(all_labels)}")

## Training Curves

Visualize how training progressed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

epochs = range(1, len(history['train_loss']) + 1)

# Loss curves
axes[0].plot(epochs, history['train_loss'], 'b-', label='Train Loss', alpha=0.8)
axes[0].plot(epochs, history['val_loss'], 'r-', label='Val Loss', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(epochs, history['train_acc'], 'b-', label='Train Acc', alpha=0.8)
axes[1].plot(epochs, history['val_acc'], 'r-', label='Val Acc', alpha=0.8)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Learning Rate Schedule

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

epochs = range(1, len(history['learning_rates']) + 1)
ax.plot(epochs, history['learning_rates'], 'g-', alpha=0.8, linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule')
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

plt.tight_layout()
plt.savefig('learning_rate_schedule.png', dpi=150, bbox_inches='tight')
plt.show()

## Training Summary

In [ ]:
print("\nTraining Summary:\n")
print(f"{'Metric':<30} {'Value':<20}")
print("="*50)
print(f"{'Best Validation Accuracy':<30} {best_val_acc:.2f}%")
print(f"{'Test Accuracy':<30} {test_acc:.2f}%")
print(f"{'Epochs Trained':<30} {len(history['train_loss'])}")
print(f"{'Final Train Accuracy':<30} {history['train_acc'][-1]:.2f}%")
print(f"{'Final Val Accuracy':<30} {history['val_acc'][-1]:.2f}%")
print(f"{'Final Train Loss':<30} {history['train_loss'][-1]:.4f}")
print(f"{'Final Val Loss':<30} {history['val_loss'][-1]:.4f}")
print(f"{'Final Learning Rate':<30} {history['learning_rates'][-1]:.2e}")
print("="*50)

## Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# Plot full confusion matrix using matplotlib instead of seaborn
plt.figure(figsize=(20, 18))
im = plt.imshow(cm, cmap='Blues', aspect='auto')
plt.colorbar(im, label='Count')

# Set ticks and labels
plt.xticks(range(len(class_names)), class_names, rotation=90, ha='right', fontsize=8)
plt.yticks(range(len(class_names)), class_names, fontsize=8)
plt.xlabel('Predicted Species', fontsize=12)
plt.ylabel('True Species', fontsize=12)
plt.title(f'Confusion Matrix (Test Accuracy: {test_acc:.2f}%)', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix_full.png', dpi=150, bbox_inches='tight')
plt.show()

# Find misclassified samples
misclassified = []
for i, (true, pred) in enumerate(zip(all_labels, all_preds)):
    if true != pred:
        misclassified.append({
            'true': class_names[true],
            'predicted': class_names[pred]
        })

print(f"\nTotal misclassifications: {len(misclassified)} out of {len(all_labels)}")
print(f"Error rate: {100 * len(misclassified) / len(all_labels):.2f}%")

if len(misclassified) > 0:
    print("\nMisclassified samples:")
    for i, error in enumerate(misclassified, 1):
        print(f"{i}. True: {error['true']:30s} -> Predicted: {error['predicted']}")

## Per-Class Performance

In [ ]:
# Classification report for model
report = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True, zero_division=0)

# Extract per-class metrics
class_metrics = []
for class_name in class_names:
    if class_name in report:
        class_metrics.append({
            'class': class_name,
            'precision': report[class_name]['precision'] * 100,
            'recall': report[class_name]['recall'] * 100,
            'f1-score': report[class_name]['f1-score'] * 100,
            'support': report[class_name]['support']
        })

# Sort by f1-score
class_metrics.sort(key=lambda x: x['f1-score'], reverse=True)

# Plot top 10 and bottom 10 performing classes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Top 10
top_10 = class_metrics[:10]
names_top = [m['class'] for m in top_10]
f1_top = [m['f1-score'] for m in top_10]
ax1.barh(names_top, f1_top, color='green', alpha=0.6)
ax1.set_xlabel('F1-Score (%)')
ax1.set_title('Top 10 Best Performing Species')
ax1.grid(True, alpha=0.3, axis='x')
ax1.invert_yaxis()

# Bottom 10
bottom_10 = class_metrics[-10:]
names_bottom = [m['class'] for m in bottom_10]
f1_bottom = [m['f1-score'] for m in bottom_10]
ax2.barh(names_bottom, f1_bottom, color='red', alpha=0.6)
ax2.set_xlabel('F1-Score (%)')
ax2.set_title('Bottom 10 Performing Species')
ax2.grid(True, alpha=0.3, axis='x')
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('per_class_performance.png', dpi=150, bbox_inches='tight')
plt.show()

# Print detailed metrics
print("\nPer-Class Performance Metrics:\n")
print(f"{'Class':<35} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<8}")
print("="*85)
for m in class_metrics:
    print(f"{m['class']:<35} {m['precision']:>10.2f}% {m['recall']:>10.2f}% {m['f1-score']:>10.2f}% {m['support']:>6.0f}")

print("\nOverall Metrics:")
print(f"Accuracy: {report['accuracy'] * 100:.2f}%")
print(f"Macro avg precision: {report['macro avg']['precision'] * 100:.2f}%")
print(f"Macro avg recall: {report['macro avg']['recall'] * 100:.2f}%")
print(f"Macro avg f1-score: {report['macro avg']['f1-score'] * 100:.2f}%")

## Save Final Results

In [ ]:
# Compile all results
final_results = {
    'model': 'EfficientNet-B0',
    'configuration': {
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'focal_alpha': FOCAL_ALPHA,
        'focal_gamma': FOCAL_GAMMA,
        'patience': PATIENCE
    },
    'validation_results': {
        'best_val_acc': float(best_val_acc)
    },
    'test_results': {
        'test_acc': float(test_acc),
        'total_samples': len(all_labels),
        'correct_predictions': sum(1 for t, p in zip(all_labels, all_preds) if t == p),
        'misclassifications': len(misclassified)
    },
    'training_stats': {
        'epochs_trained': len(history['train_loss']),
        'final_train_acc': float(history['train_acc'][-1]),
        'final_val_acc': float(history['val_acc'][-1]),
        'final_train_loss': float(history['train_loss'][-1]),
        'final_val_loss': float(history['val_loss'][-1]),
        'final_lr': float(history['learning_rates'][-1])
    },
    'misclassified_samples': misclassified
}

# Save to json
with open('efficientnet_final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("Results saved to efficientnet_final_results.json")
print("="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
print(f"Validation Accuracy: {best_val_acc:.2f}%")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Misclassified: {len(misclassified)} / {len(all_labels)} samples")
print(f"Model saved to: {model_filename}")
print("="*60)

## Save Final Results

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# set random seeds for reproducibility
RANDOM_STATE = 8
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# check what device we're using
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"Using device: CUDA")
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print(f"Using device: MPS (Apple Silicon)")
else:
    DEVICE = torch.device('cpu')
    print(f"Using device: CPU")

print(f"PyTorch version: {torch.__version__}")


AttributeError: partially initialized module 'pandas' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)